In [1]:
import pandas as pd

In [2]:
emocare = pd.read_csv("2_emocare_36.csv")
personas = pd.read_csv("3_personas_merged.csv")
user_types = pd.read_csv("4_user_types.csv")

emocare.shape, personas.shape, user_types.shape

((36, 3), (36, 2), (4, 2))

In [3]:
assert len(emocare) == len(personas), "emocare and personas must be row-aligned"


def build_persona_prompt(persona_text, user_type, user_type_desc, instruction, emocare_input):
    return (
        f"Instruction: \"{instruction.strip()}\"\n\n"
        f"{persona_text.strip()}\n\n"
        f"User Type: {user_type}\n"
        f"{user_type_desc.strip()}\n\n"
        f"Input: {emocare_input.strip()}"
    )


rows = []

for i in range(len(emocare)):
    persona_text = personas.loc[i, "persona"]

    instruction = emocare.loc[i, "instruction"]
    emocare_input = emocare.loc[i, "input"]
    output = emocare.loc[i, "output"]

    for _, ut_row in user_types.iterrows():
        user_type = ut_row["User Type"]
        user_type_desc = ut_row["Description"]

        persona_prompt = build_persona_prompt(
            persona_text, user_type, user_type_desc, instruction, emocare_input
        )

        rows.append(
            {
                "input": emocare_input,
                "output": output,
                "user_type": user_type,
                "prompt": persona_prompt,
            }
        )
        
prompt_df = pd.DataFrame(rows)
prompt_df.to_csv("5_prompts.csv", index=False)
prompt_df.shape

(144, 4)

In [4]:
prompt_df.head()

,input,output,user_type,prompt
0,"As a high school teacher, I enjoy teaching kid...",It's great that you enjoy your profession and ...,Unconscious User,"Instruction: ""If you are a counsellor, please ..."
1,"As a high school teacher, I enjoy teaching kid...",It's great that you enjoy your profession and ...,Avoidant User,"Instruction: ""If you are a counsellor, please ..."
2,"As a high school teacher, I enjoy teaching kid...",It's great that you enjoy your profession and ...,Enthusiast User,"Instruction: ""If you are a counsellor, please ..."
3,"As a high school teacher, I enjoy teaching kid...",It's great that you enjoy your profession and ...,Informed User,"Instruction: ""If you are a counsellor, please ..."
4,I'm worried that I'm not doing well in my grad...,"It's normal to feel uncertain at times, but I'...",Unconscious User,"Instruction: ""If you are a counsellor, please ..."


In [6]:
print(prompt_df["prompt"][0])

Instruction: "If you are a counsellor, please answer the questions based on the description of the patient."

Persona: Marianus Kerketta is a 35‑year‑old early‑childhood teacher who blends folk traditions with modern pedagogy, loves cricket and photography, saves diligently while splurging on camera gear, and balances community service with personal ambition.

Professional Persona: Marianus Kerketta is a dedicated early‑childhood educator who designs interactive, nature‑based curricula for pre‑primary learners in the Andaman islands, blending storytelling, folk songs, and hands‑on activities while balancing curiosity with practical classroom management; they are reliably organized yet flexible enough to adapt lessons on the fly, and they assertively pursue senior leadership roles while staying cooperative with colleagues.

Linguistic Persona: Marianus Kerketta is fluent in Hindi, able to read, write, and converse with ease, and possesses functional English skills sufficient for draftin